In [ ]:
from tqdm import tqdm
import itertools
from functools import partial

import numpy as np
import pandas as pd
import joblib
import yaml
import skfp
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors, rdmolops, rdDepictor
from skfp.fingerprints import ECFPFingerprint, EStateFingerprint, FunctionalGroupsFingerprint


class CustomDescriptor:
    """
    Custom descriptor class for calculating various molecular descriptors.
    Attributes:
        csm_runner (CSMRunner): An instance of the CSMRunner class for symmetry analysis.
        desc (dict): A dictionary mapping descriptor names to their corresponding functions.
    Methods:
        calculate_percentage(mol: Chem.Mol, idx: int) -> float:
            Calculate the percentage of a specific atom in the molecule.
        calculate_symmetry(mol: Chem.Mol, point_group: list) -> float:
            Calculate symmetry descriptor for the molecule.
        fit_transform(smiles: list) -> np.ndarray:
            Custom descriptor for fingerprints.
        get_feature_names_out() -> list:
            Get feature names for custom descriptors.
    """

    def __init__(self):

        self.desc = {
            "diameter": skfp.descriptors.diameter,
            "diameter_filtered": self.calculate_diameter_endings,
            "num_heteroatoms": rdMolDescriptors.CalcNumHeteroatoms,
            "num_rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds,
            "mol_wt": rdMolDescriptors.CalcExactMolWt,
            "o%": partial(self.calculate_percentage, idx=8),
            "n%": partial(self.calculate_percentage, idx=7),
            "c%": partial(self.calculate_percentage, idx=6),
        }

    @staticmethod
    def calculate_percentage(mol: Chem.Mol, idx: int) -> float:
        """
        Calculate the percentage of oxygen atoms in the molecule.
        :param idx: atomic number of the atom to calculate percentage for.
        :param mol: RDKit molecule object.
        :return: percentage of oxygen atoms.
        """
        num_oxygen = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == idx)
        num_atoms = mol.GetNumAtoms()
        return num_oxygen / num_atoms

    def calculate_diameter_endings(self, mol: Chem.Mol) -> int:
        """
        Finds the maximum distance between atoms of the same pattern,
        highlights the path, and returns an SVG image.
        """
        regular_diameter = skfp.descriptors.diameter(mol)

        patterns = [
            'O=*',
            'C#N',
            'Cl-*',
            'Br-*',
            '[NH2]',
            '[HO]',
        ]
        if mol is None:
            return 0

        rdDepictor.Compute2DCoords(mol)
        dist_matrix = rdmolops.GetDistanceMatrix(mol)

        max_dist = -1

        for pattern_str in patterns:
            pattern = Chem.MolFromSmarts(pattern_str)
            matches = mol.GetSubstructMatches(pattern)
            if len(matches) < 2:
                continue

            for m1, m2 in itertools.combinations(matches, 2):
                heavy1 = [i for i in m1 if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]
                heavy2 = [i for i in m2 if mol.GetAtomWithIdx(i).GetAtomicNum() > 1]
                for i in heavy1:
                    for j in heavy2:
                        d = dist_matrix[i][j]
                        if d > max_dist:
                            max_dist = d

        if max_dist < 0:
            return regular_diameter
        return max_dist

    def fit_transform(self, smiles: list) -> np.ndarray:
        """
        Custom descriptor for fingerprints.
        :param smiles: list of SMILES strings.
        :return: array with custom descriptors.
        """
        descriptors = np.zeros((len(smiles), len(self.desc.keys())))
        for i, s in tqdm(enumerate(smiles)):
            mol = Chem.MolFromSmiles(s)
            if mol is None:
                continue
            for j, (name, func) in enumerate(self.desc.items()):
                moll = Chem.MolFromSmiles(s)
                descriptors[i, j] = func(moll)
        return descriptors

    def get_feature_names_out(self) -> list:
        """
        Get feature names for custom descriptors.
        :return: list of feature names.
        """
        return list(self.desc.keys())

    def get_feature_types(self) -> dict:
        """
        Get feature types for custom descriptors.
        :return: list of feature types.
        """
        return {"categorical": [], "binary": [], "numerical": list(self.desc.keys())}

class CustomPatterns:
    """
    Custom pattern descriptor class for calculating the presence of specific substructure patterns in molecules.
    Attributes:
        patterns (dict): A dictionary mapping pattern names to their corresponding SMARTS patterns.
    Methods:
        fit_transform(smiles: list) -> np.ndarray:
            Custom descriptor for patterns.
        get_feature_names_out() -> list:
            Get the names of the custom pattern features.
    """

    def __init__(self, count: bool = False):
        patterns = ["c2ccc1ccccc1c2", "c2ccc(c1ccccc1)cc2", "c1ccncc1", "c1ncncn1", "N#Cc1ccccc1", "C/C=C/C=C/C", "C=CC(=O)C=C"]
        self.patterns = {f"custom_pattern_{i}": pattern for i, pattern in enumerate(patterns)}
        self.count = count

    def fit_transform(self, smiles: list) -> np.ndarray:
        """
        Custom descriptor for patterns.
        :param smiles: list of SMILES strings.
        :param count: whether to count occurrences or just indicate presence.
        :return: array with custom pattern descriptors.
        """
        descriptors = np.zeros((len(smiles), len(self.patterns)))
        for i, s in enumerate(smiles):
            mol = Chem.MolFromSmiles(s)
            if mol is None:
                continue
            for j, pattern in enumerate(self.patterns.values()):
                patt_mol = Chem.MolFromSmarts(pattern)
                if mol.HasSubstructMatch(patt_mol):
                    if not self.count:
                        descriptors[i, j] = 1
                    else:
                        matches = mol.GetSubstructMatches(patt_mol)
                        descriptors[i, j] = len(matches)
        return descriptors

    def get_feature_names_out(self) -> list[str]:
        """
        Get the names of the custom pattern features.
        :return: list of feature names.
        """
        return list(self.patterns.keys())

    def get_feature_types(self) -> dict:
        """
        Get feature types for custom patterns.
        :return: list of feature types.
        """
        if self.count:
            return {"categorical": [], "binary": [], "numerical": list(self.patterns.keys())}
        return {"categorical": [], "binary": list(self.patterns.keys()), "numerical": []}


def _generate_features(smiles: str, config: dict):
    """
    Generate features for a given SMILES string.
    :param smiles: SMILES string of the molecule.
    :param config: Configuration dictionary for feature generation.
    :return: Generated features as a numpy array.
    """
    ecfp_config = config.get("ecfp", {})
    estate_config = config.get("estate", {})
    fg_config = config.get("functional_groups", {})

    ecfp = ECFPFingerprint(
        fp_size=ecfp_config.get('size', 0),
        radius=ecfp_config.get('radius', 0),
        include_chirality=True,
        count=ecfp_config.get('count', False)
    )
    estate = EStateFingerprint(variant=estate_config.get('variant', 'sum'))
    functional_groups = FunctionalGroupsFingerprint(count=fg_config.get('count', False))
    descriptor = CustomDescriptor()
    patterns = CustomPatterns(count=True)

    smiles_list = [smiles]
    ecfp_features = ecfp.fit_transform(smiles_list)
    ecfp_feature_names = [f"ecfp_{i}" for i in range(ecfp_features.shape[1])]
    estate_features = estate.fit_transform(smiles_list)
    estate_feature_names = [f"estate_{i}" for i in range(estate_features.shape[1])]
    descriptor_features = descriptor.fit_transform(smiles_list)
    descriptor_features_names = descriptor.get_feature_names_out()
    fg_features = functional_groups.fit_transform(smiles_list)
    fg_feature_names = [f"functional_groups_{i}" for i in range(fg_features.shape[1])]
    pattern_features = patterns.fit_transform(smiles_list)
    patterns_feature_names = patterns.get_feature_names_out()

    feature_names = ecfp_feature_names + estate_feature_names + descriptor_features_names + fg_feature_names+ patterns_feature_names
    features = np.concat([ecfp_features, estate_features, descriptor_features, fg_features, pattern_features], axis=1)

    features = pd.DataFrame(features, columns=feature_names)

    features_to_keep = config.get("features_to_keep", [])

    features = features[features_to_keep]
    return features

In [ ]:
import shap

def _format_feature_name(feature_name: str) -> str:
    functional_group_mapping = [
                "aliphatic carboxylic acids",
                "aliphatic hydroxyl groups",
                "aliphatic hydroxyl groups excluding tert-OH",
                "N functional groups attached to aromatics",
                "Aromatic carboxylic acide",
                "aromatic nitrogens",
                "aromatic amines",
                "aromatic hydroxyl groups",
                "carboxylic acids COO",
                "carboxylic acids COO2",
                "carbonyl O",
                "carbonyl O, excluding COOH",
                "thiocarbonyl",
                "C(OH)CCN-Ctert-alkyl or  C(OH)CCNcyclic",
                "Imines",
                "Tertiary amines",
                "Secondary amines",
                "Primary amines",
                "hydroxylamine groups",
                "XCCNR groups",
                "tert-alicyclic amines (no heteroatoms, not quinine-like bridged N)",
                "H-pyrrole nitrogens",
                "thiol groups",
                "aldehydes",
                "alkyl carbamates (subject to hydrolysis)",
                "alkyl halides",
                "Allylic oxidation sites excluding steroid dienone",
                "amides",
                "amidine groups",
                "anilines",
                "aryl methyl sites for hydroxylation",
                "azide groups",
                "azo groups",
                "barbiturate groups",
                "Benzene rings",
                "benzodiazepines with no additional fused rings",
                "Bicyclic",
                "diazo groups",
                "dihydropyridines",
                "epoxide rings",
                "esters",
                "ether oxygens (including phenoxy)",
                "furan rings",
                "guanidine groups",
                "halogens",
                "hydrazine groups",
                "hydrazone groups",
                "imidazole rings",
                "imide groups",
                "isocyanates",
                "isothiocyanates",
                "ketones",
                "ketones excluding diaryl, a,b-unsat. dienones, heteroatom on Calpha",
                "beta lactams",
                "cyclic esters (lactones)",
                "methoxy groups -OCH3",
                "morpholine rings",
                "nitriles",
                "nitro groups",
                "nitro benzene ring substituents",
                "non-ortho nitro benzene ring substituents",
                "nitroso groups, excluding NO2",
                "oxazole rings",
                "oxime groups",
                "para-hydroxylation sites",
                "phenols",
                "phenolic OH excluding ortho intramolecular Hbond substituents",
                "phosphoric acid groups",
                "phosphoric ester groups",
                "piperdine rings",
                "piperzine rings",
                "primary amides",
                "primary sulfonamides",
                "pyridine rings",
                "quaternary nitrogens",
                "thioether",
                "sulfonamides",
                "sulfone groups",
                "terminal acetylenes",
                "tetrazole rings",
                "thiazole rings",
                "thiocyanates",
                "thiophene rings",
                "unbranched alkanes of at least 4 members (excludes halogenated alkanes)",
                "urea groups",
            ]
    functional_group_mapping = {f'functional_groups_{i}': name for i, name in enumerate(functional_group_mapping)}
    if feature_name.startswith('estate_'):
        suffix = feature_name.removeprefix('estate_')
        return f'EState {suffix}'

    if feature_name.startswith('ecfp_'):
        suffix = feature_name.removeprefix('ecfp_')
        return f'ECFP {suffix}'

    if feature_name.startswith('functional_groups_'):
        suffix = feature_name.removeprefix('functional_groups_')
        if functional_group_mapping is None:
            return f'Functional group {suffix}'

        mapped_name = (
            functional_group_mapping.get(suffix)
            or functional_group_mapping.get(feature_name)
            or functional_group_mapping.get(int(suffix)) if suffix.isdigit() else None
        )
        return mapped_name if mapped_name else f'Functional group {suffix}'

    return feature_name

def explain_coeff(model, features):
    """
    Explain the model predictions using feature coefficients.
    :param model: Trained model with feature coefficients.
    :param features: DataFrame of features used for prediction.
    :return: DataFrame of feature importance based on coefficients.
    """
    if hasattr(model, "coef_"):
        importance = pd.DataFrame({
            "feature": [_format_feature_name(f) for f in features.columns],
            "feature_values": features.iloc[0].values,
            "importance": model.coef_[0]
        }).sort_values(by="importance", key=abs, ascending=False)
        return importance
    else:
        raise ValueError("Model does not have coefficients for explanation.")

def explain_linear(model, features, non_scaled_features, train_data):
    explainer = shap.LinearExplainer(model, train_data)
    shap_values = explainer.shap_values(features.to_numpy())

    shap_values = shap.Explanation(
        values=shap_values,
        data=non_scaled_features.to_numpy(),
        feature_names=[_format_feature_name(f) for f in features.columns]
    )
    return shap_values

def explain_shap(model, features, non_scaled_features):
    """Explain the model predictions using SHAP values."""
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(features.to_numpy())

    shap_values = shap.Explanation(
        values=shap_values,
        data=non_scaled_features.to_numpy(),
        feature_names=[_format_feature_name(f) for f in features.columns]
    )
    return shap_values

In [ ]:
from tabpfn import load_fitted_tabpfn_model


def predict_capacity(molecule_smiles: str, config_path: str) -> float:
    """
    Predict the capacity of a molecule given its SMILES representation.
    Args:
        molecule_smiles (str): SMILES string of the molecule.
        config_path (str): Path to the trained model's config.
    Returns:
        float: Predicted capacity.
    """

    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    with open(config['numerical_features'], 'r') as f:
        numerical_features = pd.read_csv(f, header=None).iloc[:, 0].tolist()

    model_path = config['model']
    scaler_path = config['scaler']
    data_config = config['dataset']

    train_data = None
    if 'train_data' in config:
        train_data_path = config['train_data']
        train_data = pd.read_csv(train_data_path)

    with open(data_config['features_to_keep'], 'r') as f:
        data_config['features_to_keep'] = pd.read_csv(f, header=None).iloc[:, 0].tolist()


    features = _generate_features(molecule_smiles, data_config)

    with open(model_path, 'rb') as f:
        model = joblib.load(f)
    with open(scaler_path, 'rb') as f:
        scaler = joblib.load(f)

    non_scaled_features = features.copy()

    features[numerical_features] = scaler.transform(features[numerical_features].values)

    predicted_capacity = model.predict(features.values)

    explain = config['explain']
    if explain == 'coefficient':
        fi = explain_linear(model, features, non_scaled_features, train_data)
    else:
        fi = explain_shap(model, features, non_scaled_features)

    return round(float(predicted_capacity[0]), 3), fi


def predict_capacity_tabpfn(molecule_smiles: str):
    """
    Predict the capacity of a molecule given its SMILES representation.
    Args:
        molecule_smiles (str): SMILES string of the molecule.
    Returns:
        float: Predicted capacity.
    """
    config_path = "tabpfn_fg_nozhu/config.yaml"

    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    model_path = config['model']
    data_config = config['dataset']

    with open(data_config['features_to_keep'], 'r') as f:
        data_config['features_to_keep'] = pd.read_csv(f, header=None).iloc[:, 0].tolist()

    features = _generate_features(molecule_smiles, data_config)

    model = load_fitted_tabpfn_model(model_path, device="cpu")

    predicted_capacity = model.predict(features.values)

    return round(float(predicted_capacity[0]), 3), None


In [ ]:
smiles_df = pd.read_csv('lab_smiles.csv')
smiles = smiles_df['smiles'].tolist()

In [ ]:
capacitance_pred = []
config_path = 'brr_best_model/config.yaml'
save_file_path = 'brr_best_model/'
fis = {}
for smi in smiles:
    pred, fi = predict_capacity(smi, config_path)
    fis[smi] = fi
    capacitance_pred.append(pred)

In [ ]:
capacitance_pred = pd.DataFrame(
    {
        'SMILES': smiles,
        'Predicted_Capacitance': capacitance_pred
    }
)
capacitance_pred

In [ ]:
pred_cap_with_real = capacitance_pred.copy()
pred_cap_with_real['lab'] = smiles_df['lab'].values
pred_cap_with_real.sort_values(by='lab', ascending=False)

In [ ]:
real_cap = smiles_df['lab'].values

def smape(y_true, y_pred):
    epsilon = np.full(y_true.shape, 1e-10)
    return np.mean(2 * np.abs(y_pred - y_true) / (np.maximum(np.abs(y_true) + np.abs(y_pred), epsilon)))

def pa(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Calculates average ranking score given predictions and ground truth labels.
    :param y_true: Ground truth labels.
    :param y_pred: Predictions.
    :return: Average ranking score.
    """
    df = pd.DataFrame({"Test": y_true, "Pred": y_pred})
    sorted_rank = df.sort_values(by="Test", ascending=False).reset_index(drop=True)
    sorted_rank["rank"] = sorted_rank.index + 1

    cartesian_df = sorted_rank.merge(sorted_rank, how="cross", suffixes=("_1", "_2"))
    cartesian_df = cartesian_df.loc[cartesian_df["Test_1"] != cartesian_df["Test_2"]]
    cartesian_df["target_test"] = cartesian_df["Test_1"] > cartesian_df["Test_2"]
    cartesian_df["target_pred"] = cartesian_df["Pred_1"] > cartesian_df["Pred_2"]
    cartesian_df["result"] = cartesian_df["target_test"] == cartesian_df["target_pred"]

    return cartesian_df["result"].mean()

print(f'SMAPE: {smape(np.array(real_cap), np.array(capacitance_pred["Predicted_Capacitance"])):.4f}')
print(f'PA: {pa(np.array(real_cap), np.array(capacitance_pred["Predicted_Capacitance"])):.4f}')

In [ ]:
import os

capacitance_pred['real_capacitance'] = real_cap
capacitance_pred.to_excel(os.path.join(save_file_path, 'predicted_capacitance_results_current.xlsx'), index=False)

In [ ]:
import copy
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit.Chem.Draw import rdMolDraw2D

def show_molecule(smiles: str, annot: str, width: int = 400, height: int = 300) -> str:
    """
    Render a molecule from SMILES as SVG and include `annot` as the legend.

    Args:
        smiles: SMILES string of the molecule.
        annot: Text to show in the RDKit legend.
        width: SVG width.
        height: SVG height.

    Returns:
        SVG markup as a string.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")

    rdDepictor.Compute2DCoords(mol)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    drawer.DrawMolecule(mol, legend=annot)
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()

    return svg


def visualize_importance_shap(fi):
    """
    Visualize feature importance using SHAP values.
    :param fi: SHAP values for the features.
    :param features: DataFrame of features used for prediction.
    """
    # Use the first explanation (single sample) and flatten arrays
    expl = fi[0]
    shap_vals = np.ravel(np.array(expl.values))
    feature_data = np.ravel(np.array(expl.data))
    feature_names = list(expl.feature_names)

    # select top 10 features by absolute SHAP value
    abs_importance = np.abs(shap_vals)
    if abs_importance.size == 0:
        raise ValueError("No SHAP values found in explanation object")
    top_idx = np.argsort(-abs_importance)[:10]

    # preserve the ordering by abs importance descending for display
    top_idx_ordered = top_idx[np.argsort(-abs_importance[top_idx])]

    # split indices into present and absent (preserve top-order)
    present_idx = [i for i in top_idx_ordered if feature_data[i] != 0]
    absent_idx = [i for i in top_idx_ordered if feature_data[i] == 0]

    # create small Explanation objects for each subset so we can use shap.plots.bar
    def _make_expl(indices):
        if len(indices) == 0:
            return None
        vals = np.array(expl.values)[indices]
        data = np.array(expl.data)[indices]
        names = [feature_names[i] for i in indices]
        return shap.Explanation(values=vals, data=data, feature_names=names)

    present_expl = _make_expl(present_idx)
    absent_expl = _make_expl(absent_idx)

    # plotting using SHAP's bar style with larger canvas to avoid annotation collisions
    fig_height = max(4, 0.6 * len(top_idx))
    fig, axes = plt.subplots(1, 2, figsize=(18, fig_height))

    # left: present features
    if present_expl is None:
        axes[0].text(0.5, 0.5, 'No features', ha='center', va='center', fontsize=12)
        axes[0].set_axis_off()
    else:
        shap.plots.bar(present_expl, max_display=len(present_idx), show=False, ax=axes[0])
        # reduce font size of bar value annotations to prevent collisions
        for text in axes[0].texts:
            text.set_fontsize(8)

    # right: absent features
    if absent_expl is None:
        axes[1].text(0.5, 0.5, 'No features', ha='center', va='center', fontsize=12)
        axes[1].set_axis_off()
    else:
        shap.plots.bar(absent_expl, max_display=len(absent_idx), show=False, ax=axes[1])
        # reduce font size of bar value annotations to prevent collisions
        for text in axes[1].texts:
            text.set_fontsize(8)

    axes[0].set_title(f'Top present features (of top {len(top_idx)})')
    axes[1].set_title(f'Top absent features (of top {len(top_idx)})')

    # adjust layout to prevent collisions: larger margins, more space between panels
    plt.subplots_adjust(wspace=0.35, left=0.22, right=0.96, top=0.90, bottom=0.10)
    return fig, axes

def visualize_importance_coeff(fi):
    """
    Visualize feature importance based on model coefficients.
    :param fi: DataFrame of feature importance based on coefficients.
    """
    plt.figure(figsize=(10, 6))
    sns.barplot(x="importance", y="feature", data=fi.head(20))
    plt.title("Top 20 Feature Importances Based on Coefficients")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    return plt.gca()

In [ ]:
from IPython.core.display import SVG
import os

for i, (smi, fi) in enumerate(fis.items()):
    smi_path = f"smiles_{i}"
    os.makedirs(smi_path, exist_ok=True)
    predicted_cap = capacitance_pred[capacitance_pred['SMILES'] == smi].iloc[0]
    pred = predicted_cap['Predicted_Capacitance']
    real = predicted_cap['real_capacitance']
    print(f"Feature importance for SMILES: {smi}")
    svg = show_molecule(smi, annot=f'predicted: {pred}, real: {real}')
    # with open(f'{smi_path}/mol.svg', 'w') as f:
    #         f.write(svg)
    display(SVG(svg))
    fig, axes = visualize_importance_shap(fi)

    plt.show()
    print("\n")